# 12 — Spring Cloud

**What you'll learn:**

- Why Spring Cloud — what changes when one service becomes many
- The Spring Cloud umbrella — which sub-projects matter today
- **Centralised configuration** with **Spring Cloud Config Server** + client
- **Service discovery** with **Eureka** — why discovery, and how it works
- **API Gateway** with **Spring Cloud Gateway** — routes, predicates, filters, rate limiting
- **Circuit breakers** with **Resilience4j** — `@CircuitBreaker`, `@Retry`, `@TimeLimiter`, fallbacks
- **Distributed tracing** with **Micrometer Tracing** + **OpenTelemetry** — trace context propagation across hops
- **Spring Cloud Stream** — the binder abstraction that decouples your messaging code from Kafka or RabbitMQ
- Where Spring Cloud overlaps with the platform (Kubernetes, Istio) — and what you still need

This is the final notebook of the Java track. It turns the production-ready Spring service from notebook 11 into one node of a distributed system.

## Why Spring Cloud

A single Spring Boot service runs on one machine, reads its config from one file, talks directly to its database, and is one HTTP hop from its caller. The moment you split that service into many — payments, orders, fulfilment, users, notifications — every assumption breaks:

- **Where is each service?** Host names drift, instances scale up and down, a load balancer in the middle is fragile.
- **Where does config live?** Editing one YAML in one repo doesn't work when 30 services need consistent feature flags.
- **What happens when one service is slow?** Without protection, slow callees can cascade and take everything down.
- **Why is this request slow?** It crossed five services. Logs in each are isolated. Where did the time go?
- **How does the message bus stay portable?** The producer code shouldn't know whether Kafka or RabbitMQ is behind it.

Spring Cloud is an **umbrella of projects** that solve these problems with the same idiomatic Spring Boot patterns you already know. Add a starter, configure a few properties, annotate a few classes — that's the developer experience.

## What's in the umbrella (and what to skip)

Spring Cloud has accreted a lot over the years. Some projects are essential, some are legacy, and some have been superseded by platform-level tools like Kubernetes and Istio. The projects worth your attention in 2026:

| Project | Purpose | Status |
|---|---|---|
| **Spring Cloud Config** | externalise configuration to a central server | core |
| **Spring Cloud Netflix Eureka** | service discovery registry | core for VMs, optional on Kubernetes |
| **Spring Cloud Gateway** | edge / API gateway | core |
| **Spring Cloud Circuit Breaker** | wraps Resilience4j (and others) | core |
| **Spring Cloud Stream** | broker-agnostic messaging | core if you want portability |
| **Spring Cloud OpenFeign** | declarative HTTP clients | optional — `RestClient` is fine |
| **Spring Cloud Sleuth** | distributed tracing | **deprecated** — use Micrometer Tracing |
| **Spring Cloud Hystrix** | older circuit breaker | **deprecated** — use Resilience4j |
| **Spring Cloud Ribbon** | client-side load balancing | **deprecated** — use Spring Cloud LoadBalancer |

We'll cover the *core* row. The deprecated rows are still alive in old codebases, so it's worth recognising the names.

## Spring Cloud Config — one source of truth for configuration

Spring Cloud Config Server is a small Spring Boot app that serves configuration to other services. The configs themselves live in a Git repository — versioned, code-reviewed, audited. Each client service asks the config server for *its* config at startup (and optionally subscribes to refresh events).

```text
   ┌────────────────────┐         ┌────────────────────────┐
   │  Git repo          │ <────── │  Config Server         │
   │   payment.yml      │         │   (Spring Boot app)    │
   │   payment-prod.yml │         └───────────┬────────────┘
   │   order.yml        │                     │ HTTP
   │   ...              │                     │
   └────────────────────┘                     │
                                  ┌───────────┴───────────┐
                                  │                       │
                          ┌───────▼──────┐       ┌────────▼──────┐
                          │ payment-svc  │       │ order-svc     │
                          └──────────────┘       └───────────────┘
```

The server is just a Boot app with one extra annotation:

In [ ]:
// the Config Server itself
@SpringBootApplication
@EnableConfigServer
public class ConfigServerApp {
    public static void main(String[] args) {
        SpringApplication.run(ConfigServerApp.class, args);
    }
}

// application.yml on the server
server:
  port: 8888
spring:
  cloud:
    config:
      server:
        git:
          uri: https://github.com/myorg/app-configs
          default-label: main

In [ ]:
// a CLIENT service — points its bootstrap at the config server
spring:
  application:
    name: payment-svc                          # used to look up payment-svc.yml
  config:
    import: "configserver:http://config-server:8888"
  profiles:
    active: prod                                # also pulls payment-svc-prod.yml

On startup, the client asks `http://config-server:8888/payment-svc/prod` and gets back a merged property map. Spring layers it into the existing property source chain (notebook 11) at high precedence.

**Refresh without restart.** Annotate beans that should re-bind config with `@RefreshScope`. POST to `/actuator/refresh` on the client (or wire up Spring Cloud Bus to broadcast refreshes over Kafka/RabbitMQ) and the bean is rebuilt with the new values.

**The pattern.** Configuration is code. Commit it, review it, deploy it through the same pipeline as your services. The config server is just a glorified Git client with a Spring-shaped delivery surface.

## Service discovery with Eureka

When `payment-svc` needs to call `order-svc`, it shouldn't have to know the IP, port, or even how many instances exist. A **service registry** solves this: every service registers itself on startup, callers query the registry by *name*, and the registry returns the current set of healthy instances.

**Eureka** is the registry from the Netflix OSS stack that Spring Cloud has wrapped since the early days. It's a tiny Boot app that maintains an in-memory table of `{service name → list of instance URLs}`, updated by heartbeats from each registered service.

In [ ]:
// the Eureka server
@SpringBootApplication
@EnableEurekaServer
public class EurekaServerApp {
    public static void main(String[] args) { SpringApplication.run(EurekaServerApp.class, args); }
}

// application.yml on the Eureka server — single-node
server:
  port: 8761
eureka:
  client:
    register-with-eureka: false
    fetch-registry: false

In [ ]:
// CLIENT service — registers itself + uses logical names to call others
spring:
  application:
    name: payment-svc
eureka:
  client:
    service-url:
      defaultZone: http://eureka:8761/eureka

// in code — use the logical service name as the host
@Bean
public RestClient orderClient(RestClient.Builder builder) {
    return builder.baseUrl("http://order-svc").build();  // "order-svc" is the registry name
}

// Spring Cloud LoadBalancer resolves "order-svc" to a live instance at call time

**On Kubernetes**, service discovery is built in — `http://order-svc` resolves via DNS to the cluster IP of the `order-svc` Service. Eureka becomes optional, even redundant. Run Eureka when you're on plain VMs, Docker Swarm, or a hybrid cloud where DNS-based discovery isn't enough. Skip it on Kubernetes unless you have a specific reason.

## API Gateway — the edge

**Spring Cloud Gateway** is a non-blocking edge gateway. It sits in front of your microservices and handles cross-cutting concerns that don't belong in every service: routing, authentication, rate limiting, request/response rewriting, CORS, retries, header injection.

```text
                          ┌─────────────────────┐
   browser / mobile ────▶ │  Spring Cloud       │
                          │  Gateway            │
                          │   routes by path    │
                          │   auth filter       │
                          │   rate limit        │
                          └──────────┬──────────┘
                                     │
              ┌──────────────────────┼──────────────────────┐
              │                      │                      │
       ┌──────▼─────┐         ┌──────▼─────┐         ┌──────▼─────┐
       │ payment-svc│         │ order-svc  │         │ user-svc   │
       └────────────┘         └────────────┘         └────────────┘
```

A gateway is two ideas: **predicates** (which requests does this route match?) and **filters** (what do we do before forwarding / after responding?).

In [ ]:
// application.yml — routes can be declared
spring:
  cloud:
    gateway:
      routes:
        - id: payment_route
          uri: lb://payment-svc                    # lb:// = use service discovery
          predicates:
            - Path=/api/payments/**
            - Method=GET,POST
          filters:
            - RewritePath=/api/payments/(?<segment>.*), /${segment}
            - AddRequestHeader=X-Gateway, edge

        - id: order_route
          uri: lb://order-svc
          predicates:
            - Path=/api/orders/**
          filters:
            - name: RequestRateLimiter
              args:
                redis-rate-limiter.replenishRate: 100   # tokens per second
                redis-rate-limiter.burstCapacity: 200
                key-resolver: "#{@userKeyResolver}"

You can also declare routes programmatically with the fluent `RouteLocator` API for dynamic routing. Common built-in filters cover most needs:

| Filter | Use |
|---|---|
| `AddRequestHeader` / `AddResponseHeader` | inject metadata |
| `RewritePath` | strip / rewrite prefixes |
| `RequestRateLimiter` | token-bucket rate limiting (Redis-backed) |
| `CircuitBreaker` | wrap the downstream call in Resilience4j |
| `Retry` | retry on configured statuses |
| `TokenRelay` | forward OAuth2 access tokens to downstream services |

Custom filters are just `GatewayFilter` beans. The whole gateway is reactive (Project Reactor + Netty) — non-blocking under the hood, but you don't write reactive code unless you're authoring custom filters.

## Circuit breakers with Resilience4j

When `payment-svc` calls `pricing-svc` and `pricing-svc` starts timing out, every payment request stacks up waiting. Threads pile up. Eventually `payment-svc` itself falls over. The pattern that prevents this is the **circuit breaker** — once failure rates exceed a threshold, the breaker "opens" and short-circuits future calls instantly, giving the downstream time to recover.

```text
   call rate high, failures > threshold
              │
              ▼
   ┌──────────────────────────────────────────────────────────┐
   │  CLOSED  ──────────▶  OPEN  ──── after wait ────▶  HALF  │
   │   normal             reject              probe with     OPEN │
   │   traffic            instantly           one call            │
   │                                                              │
   │      ◀─────────── probe succeeds ──────────────────         │
   │      ─────────── probe fails ──────────────────▶            │
   └──────────────────────────────────────────────────────────┘
```

**Resilience4j** is the modern circuit breaker library Spring Cloud recommends. It's lightweight, functional, integrates with Spring Boot via `spring-cloud-starter-circuitbreaker-resilience4j`, and provides four resilience primitives that compose: **CircuitBreaker**, **Retry**, **TimeLimiter**, and **Bulkhead**.

In [ ]:
# application.yml — declarative configuration
resilience4j:
  circuitbreaker:
    instances:
      pricing:
        sliding-window-size: 20
        failure-rate-threshold: 50          # %
        wait-duration-in-open-state: 30s
        permitted-number-of-calls-in-half-open-state: 3
  retry:
    instances:
      pricing:
        max-attempts: 3
        wait-duration: 200ms
        retry-exceptions:
          - java.io.IOException
  timelimiter:
    instances:
      pricing:
        timeout-duration: 2s

In [ ]:
import io.github.resilience4j.circuitbreaker.annotation.CircuitBreaker;
import io.github.resilience4j.retry.annotation.Retry;
import io.github.resilience4j.timelimiter.annotation.TimeLimiter;

@Service
public class PricingGateway {

    private final RestClient client;
    public PricingGateway(RestClient pricingClient) { this.client = pricingClient; }

    @CircuitBreaker(name = "pricing", fallbackMethod = "cachedPrice")
    @Retry(name = "pricing")
    @TimeLimiter(name = "pricing")
    public CompletableFuture<Price> fetchPrice(String sku) {
        return CompletableFuture.supplyAsync(() ->
            client.get().uri("/prices/{sku}", sku).retrieve().body(Price.class));
    }

    // fallback — same signature + a Throwable argument
    public CompletableFuture<Price> cachedPrice(String sku, Throwable ex) {
        return CompletableFuture.completedFuture(Price.fallback(sku));
    }
}

**The pattern.** Wrap every cross-service call in a circuit breaker. Provide a fallback that returns something sensible — a cached value, a default, a degraded mode. Your service stays up when its dependencies don't. Resilience4j exports rich metrics through Micrometer, so you can dashboard breaker state, success rates, and call durations alongside your normal HTTP metrics.

## Distributed tracing — Micrometer Tracing + OpenTelemetry

A request that crosses five services produces logs in five different places. Without correlation, debugging a slow request means digging through five log files and trying to line up timestamps. **Distributed tracing** stitches the hops together by propagating a **trace context** — a trace ID plus per-hop span IDs — through every call.

```text
   request id = abc                       │ trace abc
                                          │
   ┌─────────────┐  X-B3-TraceId: abc    │ ┌──── span 1 ────┐
   │  gateway    │ ──────────────────▶   │ │ gateway        │ 5ms
   └─────┬───────┘                       │ └────────────────┘
         │                               │   ┌── span 2 ──┐
         ▼                               │   │ payment    │ 80ms
   ┌─────────────┐  X-B3-TraceId: abc    │   └────────────┘
   │  payment    │ ──────────────────▶   │     ┌── span 3 ──┐
   └─────┬───────┘                       │     │ pricing    │ 60ms
         │                               │     └────────────┘
         ▼                               │
   ┌─────────────┐                       │
   │  pricing    │                       │   ◀── full timeline visible
   └─────────────┘                       │       in Tempo / Jaeger / Zipkin
```

Modern Spring uses **Micrometer Tracing** as the facade (replacing the deprecated Spring Cloud Sleuth) and **OpenTelemetry** or **Zipkin Brave** as the backend implementation. Plug it in once and traces flow through `RestClient`, `WebClient`, `KafkaTemplate`, JDBC, and `@KafkaListener` automatically.

In [ ]:
<!-- pom.xml — pick the OTel bridge for OpenTelemetry-native back ends -->
<dependency>
    <groupId>io.micrometer</groupId>
    <artifactId>micrometer-tracing-bridge-otel</artifactId>
</dependency>
<dependency>
    <groupId>io.opentelemetry</groupId>
    <artifactId>opentelemetry-exporter-otlp</artifactId>
</dependency>

In [ ]:
# application.yml
management:
  tracing:
    sampling:
      probability: 1.0                # 100% in dev; ~5–10% in prod
  otlp:
    tracing:
      endpoint: http://otel-collector:4318/v1/traces

# add the trace ID to every log line
logging:
  pattern:
    level: "%5p [%X{traceId:-},%X{spanId:-}]"

With the bridge in place and the pattern updated, every log line includes the trace ID, every outgoing call propagates the W3C `traceparent` header (or B3 headers, configurable), and your back end of choice (Tempo, Jaeger, Honeycomb, Datadog) shows the full request waterfall.

You almost never write tracing code by hand. The instrumentation is implicit. When you do want to add a custom span — say, around a slow internal computation — use the Micrometer `Observation` API or `@NewSpan` from `micrometer-tracing-annotation`.

## Spring Cloud Stream — broker-agnostic messaging

Notebook 11 covered `spring-kafka` directly. That works, but it ties your producer and consumer code to Kafka. If you ever switch to RabbitMQ, or want to run the same code on a Kafka-mock in tests, you rewrite everything.

**Spring Cloud Stream** is a higher-level abstraction. You write messaging logic as plain `java.util.function` types — `Supplier`, `Function`, `Consumer` — and a **binder** plugs them into the actual broker. Swap binders by swapping a dependency; the business code doesn't change.

In [ ]:
// the message handlers are just functions
@Configuration
public class OrderFlow {

    // a Supplier becomes a producer — every emit publishes to the output binding
    @Bean
    public Supplier<OrderEvent> orderProducer() {
        return () -> nextOrder();          // polled periodically; or use StreamBridge for on-demand
    }

    // a Consumer becomes a sink — invoked per inbound message
    @Bean
    public Consumer<OrderEvent> orderSink() {
        return event -> log.info("got order {}", event.id());
    }

    // a Function becomes a stream processor — read from one binding, write to another
    @Bean
    public Function<OrderEvent, ShipmentEvent> orderToShipment() {
        return order -> new ShipmentEvent(order.id(), Instant.now());
    }
}

In [ ]:
# application.yml — bind function names to topics, plus pick the broker
spring:
  cloud:
    stream:
      bindings:
        orderProducer-out-0:
          destination: orders
        orderSink-in-0:
          destination: orders
          group: order-loggers
        orderToShipment-in-0:
          destination: orders
        orderToShipment-out-0:
          destination: shipments
      # swap binder — kafka, rabbit, kinesis, gcp-pubsub, ...
      kafka:
        binder:
          brokers: localhost:9092

**The trade.** You get portability, a consistent programming model across brokers, and a clean test story (use the test binder, no broker required). You give up direct access to broker-specific features (Kafka's transactions, RabbitMQ's headers exchange routing) — you can break out to the native API when you need them.

For new code, Cloud Stream is the cleaner shape. For existing `spring-kafka` codebases, the cost of migration usually isn't worth it.

## The platform overlap — where Spring Cloud meets Kubernetes

A lot of what early Spring Cloud built — service discovery, config distribution, secret rotation, network-level load balancing — now ships in your container platform. On Kubernetes:

- **Service discovery** → built into the `Service` resource via DNS
- **Config distribution** → `ConfigMap` and `Secret`, mounted as files or env vars
- **Load balancing** → handled by `kube-proxy` and the `Service` abstraction
- **Mutual TLS / traffic shaping** → service meshes like Istio or Linkerd

What Spring Cloud still owns at the *application* layer:

- **Spring Cloud Gateway** — application-aware routing, JWT handling, token relay
- **Resilience4j** — in-process circuit breakers, retries, time limits, bulkheads
- **Micrometer Tracing** — in-process trace context propagation
- **Spring Cloud Stream** — broker-agnostic messaging abstraction
- **Spring Cloud Config** — when you want application-level config with rich refresh semantics

The pragmatic answer: lean on Kubernetes for infrastructure, use Spring Cloud for the patterns that belong in the application.

## Track wrap-up

You have walked the full Java + Spring track:

- **Notebooks 01–07** — modern Java, from `var` and records to virtual threads, ending with the JVM internals and annotation-plus-reflection mechanics that Spring runs on.
- **Notebook 08 — Spring Core** — the IoC container, dependency injection, AOP, the application context.
- **Notebook 09 — Spring Web** — REST controllers, validation, error handling, Spring Security essentials.
- **Notebook 10 — Spring Data** — JPA entity mapping, repositories, transactions, migrations.
- **Notebook 11 — Spring Boot in Production** — auto-configuration, externalised config, Actuator, testing slices, Testcontainers, Kafka, containerised deployment.
- **Notebook 12 — Spring Cloud** — config server, service discovery, the gateway, circuit breakers, distributed tracing, broker-agnostic messaging.

You can now build, test, observe, and operate a production Spring service, and connect many of them into a coherent distributed system.

**Where to go from here.** The Java foundation generalises — `java.util.function`, the `Stream` API, virtual threads, and the JVM tooling are the same wherever you go. The Spring patterns generalise too — dependency injection, `@Transactional`, AOP-as-proxies, conditional configuration, and the convention-over-configuration philosophy show up across every Spring-shaped library, including Spring Batch, Spring Integration, Spring AI, and any future Spring project. The mental model you built is portable to all of them.

Welcome to working Java + Spring. Build something real.